# EAA Pose Pipeline — Google Colab

This notebook runs the full RGB → NTU-120 3-D 25-joint skeleton pipeline on Google Drive data.

**Steps:**
1. Mount Drive & clone/pull repo
2. Install dependencies (MMPose stack)
3. Module 1 — Filter PKU-MMD v1 interactions (CPU, ~4 min)
4. Module 2 — Pose estimation on PKU v1 / PKU v2 / TSU (GPU)

**Runtime:** T4 GPU recommended.  Switch via Runtime → Change runtime type.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = '/content/Skeleton-EAA-Pose'
REPO_URL = 'https://github.com/tuan8p/Skeleton-EAA-Pose.git'  # update if needed

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

In [ ]:
# Base deps (numpy, openpyxl, PyYAML, scipy, tqdm, opencv)
!pip install -q -r requirements.txt

# PyTorch — Colab usually has this pre-installed
import torch
print('CUDA available:', torch.cuda.is_available())
print('CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')

In [ ]:
# MMPose ecosystem
!pip install -q -U openmim
!mim install -q mmengine
!mim install -q 'mmcv>=2.1'
!mim install -q 'mmdet>=3.2'
!mim install -q 'mmpose>=1.3'
!pip install -q lap   # ByteTrack dependency

In [ ]:
# Download model checkpoints (RTMDet + RTMW3D)
import os
os.makedirs('checkpoints', exist_ok=True)

DET_CKPT = 'checkpoints/rtmdet_nano_8xb32-100e_coco-obj365-person-05d8511e.pth'
POSE_CKPT = 'checkpoints/rtmw3d-l_8xb64_cocktail14-384x288-794dbc78_20231122.pth'

if not os.path.exists(DET_CKPT):
    !mim download mmdet --config rtmdet_nano_320-8xb32_coco-person \
        --dest checkpoints/

if not os.path.exists(POSE_CKPT):
    !mim download mmpose --config rtmw3d-l_8xb64_cocktail14-384x288 \
        --dest checkpoints/

## 1. Module 1 — PKU-MMD v1 Interaction Filter

Reads original skeleton files from Drive, outputs filtered labels + Actions.xlsx.
**CPU only, ~4 minutes.**

In [ ]:
# --- Edit these paths to match your Drive layout ---
DRIVE_BASE = '/content/drive/MyDrive/ĐACN-TN_datasets/ĐATN/rawdatasets'

SKEL_DIR        = f'{DRIVE_BASE}/skeletons/PKU/Skeleton'
LABEL_DIR       = f'{DRIVE_BASE}/skeletons/PKU/Label_PKUMMD_v1'
SRC_ACTIONS     = f'{DRIVE_BASE}/skeletons/PKU/Actions.xlsx'

OUT_LABEL_DIR   = f'{DRIVE_BASE}/skeletons/PKU/Label_PKUMMDv1_daily'
OUT_ACTIONS     = f'{DRIVE_BASE}/skeletons/PKU/Actions_daily_v1.xlsx'
MANIFEST        = f'{DRIVE_BASE}/skeletons/PKU/manifest_v1.csv'

In [ ]:
!python -m eaa_pose.filter_pku_interactions \
    --config configs/pku_v1.yaml \
    --skeleton-dir     "{SKEL_DIR}" \
    --label-dir        "{LABEL_DIR}" \
    --src-actions-xlsx "{SRC_ACTIONS}" \
    --out-label-dir    "{OUT_LABEL_DIR}" \
    --out-actions-xlsx "{OUT_ACTIONS}" \
    --manifest         "{MANIFEST}"

## 2. Module 2 — Pose Estimation

Runs RTMDet → ByteTrack → RTMW3D → 25-joint mapping → QC → per-sample .npy.
**GPU recommended.**

In [ ]:
# PKU v1 — update checkpoint paths if downloaded to a different location
PKU1_VIDEO_DIR  = f'{DRIVE_BASE}/videos/PKUMMD/Data/RGB_VIDEO'
PKU1_SEG_DIR    = OUT_LABEL_DIR
PKU1_ACTIONS    = OUT_ACTIONS
PKU1_OUT_DIR    = '/content/drive/MyDrive/ĐACN-TN_datasets/ĐATN/skeletons_25/PKU_v1'

In [ ]:
!python -m eaa_pose.run_pose \
    --config configs/pku_v1.yaml \
    --video-dir    "{PKU1_VIDEO_DIR}" \
    --segments-dir "{PKU1_SEG_DIR}" \
    --actions-xlsx "{PKU1_ACTIONS}" \
    --out-dir      "{PKU1_OUT_DIR}" \
    --device cuda

In [ ]:
# TSU
TSU_VIDEO_DIR = f'{DRIVE_BASE}/videos/TSU/Videos_mp4'
TSU_SEG_DIR   = f'{DRIVE_BASE}/skeletons/TSU/Annotation_v1.0'
TSU_OUT_DIR   = '/content/drive/MyDrive/ĐACN-TN_datasets/ĐATN/skeletons_25/TSU'

!python -m eaa_pose.run_pose \
    --config configs/tsu.yaml \
    --video-dir    "{TSU_VIDEO_DIR}" \
    --segments-dir "{TSU_SEG_DIR}" \
    --out-dir      "{TSU_OUT_DIR}" \
    --device cuda

## 3. Quick sanity check

In [ ]:
import numpy as np, glob, os

samples = glob.glob(os.path.join(PKU1_OUT_DIR, '*.npy'))
print(f'PKU v1 samples: {len(samples)}')
if samples:
    arr = np.load(samples[0])
    print(f'  Shape: {arr.shape}  (T, M=1, 25 joints, 6 channels)')
    print(f'  Channels: x, y, z, confidence, valid_mask, reconstructed_flag')
    print(f'  valid_mask mean: {arr[..., 4].mean():.3f}')
    print(f'  reconstructed mean: {arr[..., 5].mean():.3f}')